In [ ]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.preprocessing import MinMaxScaler
from scipy.cluster import hierarchy 
%matplotlib inline

- First-batch model: connectivity coefficient matrix across all cell types

In [ ]:
projectpath = r'path to your projectpath fold'  # Project root containing the signal_data/, behavioral_data/, summary/, and correlation_analysis/ folders used below.
test = 'test1'
event_type = 'S_to_I'#S_to_I,I_to_S
all_file_path = {}
out_path = os.path.join(projectpath, 'correlation_analysis', test)
if not os.path.exists(out_path):os.makedirs(out_path)


all_celltypes = ['CamkII_TST','PV','SST','VIP']#'CamkII_TST', 'SST', 'PV', 'VIP'
subjects = np.arange(1, 11)  # Assume each cell type has 10 subjects (consistent with CamkII)

for i,celltype in enumerate(all_celltypes):
    all_file_path[celltype] = []
    for subject in subjects:
        subject_dir = os.path.join(projectpath, 'summary', test, celltype, str(subject))
        if os.path.exists(subject_dir):
            file_path = os.path.join(subject_dir,'signal_save', f'{event_type}_event_average_zscore_denoised.csv')
            if os.path.isfile(file_path):
                all_file_path[celltype].append(file_path)
            else:
                raise FileNotFoundError(f"File does not exist: {file_path}")
        else:
            print(f"Skipping {celltype}{subject}...")#Mouse not available




all_celltype_df = []
for i,celltype in enumerate(all_file_path.keys()):
    celltype_df = []
    for file in all_file_path[celltype]:
        subject_df = pd.read_csv(file)
        if  subject_df.shape[1] > 400:
            raise ValueError(f"DataFrame has more than 400 frames: {file}")
        subject_df = subject_df.drop(columns='Unnamed: 0')
        if subject_df.isna().any().any():
            raise ValueError(f'DataFrame has nan values: {file}')
        subject_df.insert(loc=0, column='celltype', value=i + 1)
        celltype_df.append(subject_df)
    
    celltype_df = pd.concat(celltype_df,axis=0)
    ######
    # Within-group clustering
    ########
    row_linkage = hierarchy.linkage(
    celltype_df,                # Input data (rows = samples to cluster)
    method='ward',     # Clustering method (e.g., 'ward', 'complete', 'average')
    metric='euclidean' # Distance metric (e.g., 'euclidean', 'correlation')
)
    hierarchy.dendrogram(row_linkage,orientation='left')
    row_order = hierarchy.leaves_list(row_linkage)
    # plt.show()
    # plt.close()
    celltype_df = celltype_df.iloc[row_order,:]
    temp = celltype_df.to_numpy()
    print(f'Shape of celltype{celltype}: {temp.shape}')

    
    all_celltype_df.append(celltype_df)




In [ ]:
temp

In [ ]:
maxtime = min(df.shape[1] for df in all_celltype_df)
print(f'Maxtime is {maxtime}')
all_celltype_df = [df.iloc[:,:maxtime] for df in all_celltype_df]
combined_df = pd.concat(all_celltype_df,axis=0)
temp = combined_df.copy()
scaler = MinMaxScaler()
temp.iloc[:,1:] = scaler.fit_transform(temp.iloc[:,1:].T).T
plt.imshow(temp.iloc[:,1:].to_numpy(), aspect='auto', cmap='coolwarm')
plt.colorbar(label='Z-score')
plt.title('Combined Cell Types Event Average Z-score Denoised')

In [ ]:
correlation_matrix = np.corrcoef(combined_df.drop(columns='celltype'))
print(f'correlation size{correlation_matrix.shape}')
corr_flat = correlation_matrix.flatten()
print(f'Before: max{max(corr_flat)},min:{min(corr_flat)}')

plt.hist(corr_flat, bins=50)
plt.title(f'Histogram of Correlation Values for {event_type} Events')
plt.xlabel('Correlation Coefficient')

# Step 2: Calculate mean and standard deviation of the correlation values
# mean_corr = np.mean(corr_flat)
# std_corr = np.std(corr_flat, ddof=1) 
# correlation_matrix = (correlation_matrix-mean_corr)/std_corr
# corr_flat = correlation_matrix.flatten()
# print(f'After: max{max(corr_flat)},min:{min(corr_flat)}')

# Threshold clipping
# max_median = correlation_matrix.max(axis=1).median()
# min_median = correlation_matrix.min(axis=1).median()
# min_min = correlation_matrix.min(axis=1).min()
# max_max = correlation_matrix.max(axis=1).max()




In [ ]:
# Heatmap colorbar
index = np.linspace(0, 1, 8)
colors = [(0,'#3333b2'),
            (index[1],'#3333b2'),
            (index[2],'#337fff'),
            (index[3],'#00b2b2'),
            (index[4],'#cccc33'),
            (index[5],'#ffb233'),
            (index[6],'#ffff19'),
            (1,'#ffff19')]
cmap = LinearSegmentedColormap.from_list('custom_colormap',colors)

# Clustering coefficient matrix
plt.figure(figsize=(10, 8))
plt.imshow(correlation_matrix, cmap=cmap)
plt.colorbar()
plt.title(f'Correlation Matrix for {event_type} Events')
plt.savefig(os.path.join(out_path, f'{event_type}_correlation_matrix.pdf'))
plt.show()

In [ ]:
# Cell ordering bar
cmap = mpl.colors.ListedColormap(['#E5D4E8','#1B7939','#2E2585','#B31529'])
plt.figure(figsize=(10, 0.2))
plt.yticks(np.arange(0,combined_df.shape[0], step=100))
celltype_array = combined_df['celltype'].to_numpy().reshape(1,-1)
plt.imshow(celltype_array, cmap=cmap, aspect='auto')
plt.savefig(os.path.join(out_path,f'{event_type}_celltype_bar.pdf'))

for i,celltype in enumerate(all_celltypes):
    print(f'Celltype {i+1}: {celltype}, count: {np.sum(combined_df["celltype"] == (i + 1))}')

In [ ]:
# 40-frame event transition map (all cell types)
if 'celltype' in combined_df.columns:
    combined_temp = combined_df.drop(columns='celltype')

max_max = combined_temp.max(axis=1).max()
min_min = combined_temp.min(axis=1).min()
max_median = combined_temp.max(axis=1).median()
min_median = combined_temp.min(axis=1).median()
cliped_df = combined_temp.clip(lower=min_median, upper=max_median)
print(cliped_df.max(axis=1).max())
#cliped_df = (cliped_df-min_median)/(max_median-min_median)

cmap = LinearSegmentedColormap.from_list('custom_colormap',colors)
fig, ax = plt.subplots(figsize=(10, 3))
cax = ax.imshow(cliped_df,cmap=cmap,aspect='auto')
#plt.savefig(os.path.join(out_path,f'{event_type}_combined_4s_heatmap.pdf'))    

plt.tight_layout()
plt.show()

g = sns.clustermap(cliped_df,
                    method='ward',
                    metric='euclidean',
                    cmap=cmap,
                    col_cluster=False,
                    row_cluster=True,  
                    figsize=(10, 3), 
                    cbar_kws={'label': 'Normalized Value'},
                    )
#plt.savefig(os.path.join(out_path,f'{event_type}_combined_4s_heatmap_clustered.pdf'))     

plt.show()           
plt.close()

In [ ]:
# Save & preview data
correlation_df = pd.DataFrame(correlation_matrix,columns=combined_df.index,index=combined_df.index)
correlation_df = pd.concat([correlation_df,combined_df['celltype']],axis=1)
correlation_df.to_csv(os.path.join(out_path,f'{event_type}_correlation_matrix.csv'))
combined_df